# Design arm · D3 — Persistent recalled predicates, design-related recalls only

Same definition as Stage 6, with two decisions fixed on 2026-10-15:
- **Exposure:** a predicate with at least one design-related recall. Its **recall date** is its earliest design-related recall, not its first recall of any kind.
- **Successor exclusion unchanged:** a citing device B is excluded if B has *any* recall, so this arm is nested within the primary analysis.

**Arms**
- any_recall: recomputation of the parent primary analysis; must reproduce ../data/expected_values.json
- design_fda: design category from FDA root cause via cause_map.py (available now; preliminary)
- design_plus_unclear_fda: bounding arm
- design_final and design_plus_unclear_final: from adjudicated codes, once data/event_cause_final.csv exists

Each arm runs on the primary edges (Rule 2), high-confidence edges (NEAR_CUE), and registered Rule 1 edges. Tuples are (persistent predicates, post-recall citations, downstream devices).

Writes data/expected_values_design.json and data/design_exposed_nodes.csv.

**2026-10-28 (D8/D10):** uses the parent's current definitions. A citation is post-recall when the citing
510(k) was **received after the predicate's relevant recall was public** (earliest recall-database posting
or Enforcement Report date among the relevant recall records); the relevant recall's initiation must be
a valid date (Stage 5 rule, so K060694's corrupted date is ignored). Edges are Rule 2 v2. The any_recall
arm must reproduce the parent's 2026-10 values.

In [1]:
import json, hashlib
from pathlib import Path
import pandas as pd
import networkx as nx
from cause_map import DESIGN, UNCLEAR

corpus = pd.read_csv("../data/corpus.csv", dtype=str)
corpus["decision_year"] = pd.to_numeric(corpus["decision_year"], errors="coerce")
year = dict(zip(corpus["k_number"], corpus["decision_year"]))
edges = pd.read_csv("../data/predicate_edges.csv", dtype=str)
edges_r1 = pd.read_csv("../data/predicate_edges_rule1.csv", dtype=str)
parent = json.load(open("../data/expected_values.json"))

links = pd.read_csv("data/event_device_links.csv", dtype=str)
events = pd.read_csv("data/recall_events.csv", dtype=str)
ANY_RECALLED = set(links["k_number"])   # successor exclusion: any recall
FINAL = Path("data/event_cause_final.csv")
final = pd.read_csv(FINAL, dtype=str) if FINAL.exists() else None
print("Adjudicated codes:", "found" if final is not None else "not yet; running FDA-mapping arms only")
# ---- 2026-10 (D8): exact-date post-recall rule, as in parent Stage 6 ----
import gzip
_D = lambda s: pd.to_datetime(s, errors="coerce")
SNAP = pd.Timestamp(json.load(open("../snapshot/SNAPSHOT.json"))["snapshot_date"])
received = {r["k_number"]: _D(r.get("date_received")) for r in json.load(gzip.open("../snapshot/510k_raw.json.gz", "rt"))}
_report = {e.get("recall_number"): _D(e.get("report_date"))
           for e in json.load(gzip.open("../snapshot/enforcement_raw.json.gz", "rt"))}
_public = {}                                   # product_res_number -> date that recall record was public
for r in json.load(gzip.open("../snapshot/recall_raw.json.gz", "rt")):
    d = [x for x in (_D(r.get("event_date_posted")), _report.get(r.get("product_res_number"), pd.NaT)) if pd.notna(x)]
    if d:
        _public[r.get("product_res_number")] = min(d)
links["init"] = pd.to_datetime(links["event_date_initiated"], format="%Y-%m-%d", errors="coerce")
links.loc[~((links["init"].dt.year >= 1976) & (links["init"] <= SNAP)), "init"] = pd.NaT   # Stage 5 validity rule
links["public"] = links["product_res_number"].map(_public)
assert parent.get("post_recall_rule") == "received_after_public", "parent pipeline is not on the 2026-10 rule"

Adjudicated codes: not yet; running FDA-mapping arms only


In [2]:
def build(edge_df):
    H = nx.DiGraph()
    H.add_nodes_from(corpus["k_number"])
    for p, d in zip(edge_df["predicate_knumber"], edge_df["device_knumber"]):
        if p in H and d in H:
            H.add_edge(p, d)
    return H

GRAPHS = {"primary": build(edges),
          "high_conf": build(edges[edges["confidence"] == "NEAR_CUE"]),
          "rule1": build(edges_r1)}

def exposure_dates(event_ids):
    """Per device, counting only the given recall events: (earliest valid initiation, earliest public date)."""
    sub = links[links["res_event_number"].isin(set(event_ids))]
    g = sub.groupby("k_number").agg(init=("init", "min"), public=("public", "min"))
    return {k: (r.init, r.public) for k, r in g.iterrows()}

def post(A_dates, B):
    """2026-10 rule: citing 510(k) received after the predicate's (relevant) recall was public."""
    init, pub = A_dates
    rec = received.get(B, pd.NaT)
    return pd.notna(init) and pd.notna(pub) and pd.notna(rec) and pub < rec

def analysis(H, dates):
    preds, cits, down = set(), 0, set()
    for A, ad in dates.items():
        if A not in H:
            continue
        for B in H.successors(A):
            if B in ANY_RECALLED:
                continue
            if post(ad, B):
                preds.add(A); cits += 1; down.add(B)
    return [len(preds), cits, len(down)]

def extras(H, dates):
    first_after, max_gap = 0, 0
    for A, (init, pub) in dates.items():
        if A not in H or pd.isna(init):
            continue
        cites = [B for B in H.successors(A) if pd.notna(received.get(B, pd.NaT))]
        if cites and all(post((init, pub), B) for B in cites):
            first_after += 1
        yrs = [year.get(B) for B in H.successors(A) if pd.notna(year.get(B))]
        if yrs:
            max_gap = max(max_gap, max(yrs) - init.year)   # unchanged: years from initiation
    return first_after, int(max_gap)

In [3]:
fda_cat = dict(zip(events["res_event_number"], events["fda_category"]))
arms = {
    "any_recall": list(events["res_event_number"]),
    "design_fda": [e for e, c in fda_cat.items() if c == DESIGN],
    "design_plus_unclear_fda": [e for e, c in fda_cat.items() if c in (DESIGN, UNCLEAR)],
}
if final is not None:
    fin_cat = dict(zip(final["res_event_number"], final["final_category"]))
    arms["design_final"] = [e for e, c in fin_cat.items() if c == DESIGN]
    arms["design_plus_unclear_final"] = [e for e, c in fin_cat.items() if c in (DESIGN, UNCLEAR)]

results, DATES = {}, {}
for arm, evs in arms.items():
    dates = exposure_dates(evs)
    DATES[arm] = dates
    r = {"recall_events": len(set(evs)), "exposed_devices": len(dates)}
    for g, H in GRAPHS.items():
        r[g] = analysis(H, dates)
    r["first_after_recall"], r["max_gap_years"] = extras(GRAPHS["primary"], dates)
    results[arm] = r
    print(f"CHECKPOINT  {arm:26s} exposed {r['exposed_devices']:5d} | primary {r['primary']} | high_conf {r['high_conf']} | rule1 {r['rule1']} | first_after {r['first_after_recall']}")

CHECKPOINT  any_recall                 exposed  1691 | primary [893, 2553, 1709] | high_conf [789, 1949, 1418] | rule1 [925, 2793, 1825] | first_after 356
CHECKPOINT  design_fda                 exposed   288 | primary [168, 430, 360] | high_conf [154, 324, 286] | rule1 [175, 476, 395] | first_after 68
CHECKPOINT  design_plus_unclear_fda    exposed   712 | primary [379, 1031, 811] | high_conf [335, 791, 651] | rule1 [393, 1117, 872] | first_after 147


### Reproduction check against the parent pipeline

In [4]:
base = results["any_recall"]
checks = {"primary": [parent["persistent_predicates"], parent["post_recall_citations"], parent["downstream_devices"]],
          "high_conf": parent["sens_high_conf"],
          "rule1": parent["sens_maximal_rule"]}
for g, exp in checks.items():
    print(f"REPRODUCE  {g:9s} parent {list(exp)} | here {base[g]}  {'OK' if base[g] == list(exp) else 'MISMATCH'}")
print(f"REPRODUCE  first_after parent {parent['first_after_recall']} | here {base['first_after_recall']}")
print(f"REPRODUCE  max_gap     parent {parent['max_gap_years']} | here {base['max_gap_years']}")

REPRODUCE  primary   parent [893, 2553, 1709] | here [893, 2553, 1709]  OK
REPRODUCE  high_conf parent [789, 1949, 1418] | here [789, 1949, 1418]  OK
REPRODUCE  rule1     parent [925, 2793, 1825] | here [925, 2793, 1825]  OK
REPRODUCE  first_after parent 356 | here 356
REPRODUCE  max_gap     parent 21 | here 21


### Outputs

In [5]:
nodes = pd.DataFrame({"k_number": sorted(set(DATES["design_fda"]) | set(DATES.get("design_final", {})))})
first = lambda arm: (lambda k: DATES[arm].get(k, (pd.NaT, pd.NaT))[0])      # initiation date of the arm's recall
public = lambda arm: (lambda k: DATES[arm].get(k, (pd.NaT, pd.NaT))[1])
nodes["first_any_recall"] = nodes["k_number"].map(first("any_recall"))
nodes["first_design_recall_fda"] = nodes["k_number"].map(first("design_fda"))
nodes["design_recall_public_fda"] = nodes["k_number"].map(public("design_fda"))
if "design_final" in DATES:
    nodes["first_design_recall_final"] = nodes["k_number"].map(first("design_final"))
nodes["date_moved_later_fda"] = nodes["first_design_recall_fda"] > nodes["first_any_recall"]
nodes.to_csv("data/design_exposed_nodes.csv", index=False)
print(f"CHECKPOINT  design-exposed devices (FDA mapping) whose exposure date is later than their first recall: {int(nodes['date_moved_later_fda'].sum())}")

out = {
    "parent_snapshot_date": parent.get("snapshot_date"),
    "cause_map_sha256": hashlib.sha256(Path("cause_map.py").read_bytes()).hexdigest(),
    "definition": {"exposure": "predicate linked to >=1 design-related recall event",
                   "recall_date": "earliest valid design-related recall initiation",
                   "post_recall_rule": "citing 510(k) received after the design-related recall was public",
                   "successor_exclusion": "any recall (unchanged from primary)"},
    "tuple_order": ["persistent_predicates", "post_recall_citations", "downstream_devices"],
    "adjudicated_codes_used": final is not None,
    "arms": results,
}
Path("data/expected_values_design.json").write_text(json.dumps(out, indent=1))
print("Wrote data/expected_values_design.json")

CHECKPOINT  design-exposed devices (FDA mapping) whose exposure date is later than their first recall: 41
Wrote data/expected_values_design.json
